# 第 2 章：深度学习与神经网络——完整实验版
本 Notebook 用三个最小实验把讲义中的抽象概念变成可观察结果：计算图怎样产生梯度、梯度怎样驱动 MLP 学习、图像怎样以四维 Tensor 流过 CNN。

不要只点击“全部运行”。每段代码前先阅读目的，运行后按提示解释输出。主线是：`计算图与梯度 → MLP 训练 → CNN Shape → 统一验收`。

## 准备环境：建立可复现的实验起点

这里导入路径、数组、PyTorch、数据集与评估工具，并固定随机种子。随机种子会影响初始权重，因此它是复现实验结果的一部分；它不能保证所有硬件上的浮点结果逐位相同。

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
torch.manual_seed(42)

# 实验一：Autograd 与链式法则

**为什么做？** 神经网络训练的核心不是“调用 `backward`”，而是计算 Loss 对每个参数的导数。这个标量例子让我们在没有网络结构干扰时观察 Autograd。

`requires_grad=True` 告诉 PyTorch 记录由 `x` 出发的运算；`y.backward()` 反向遍历计算图；结果累积到 `x.grad`。

**运行前预测：** 对 $y=x^2+x$ 求导，在 $x=2$ 时应为 5；对 $z=x^2+x^2/2$ 求导则应为 6。运行后确认两项手算结果。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + x
y.backward()
print('y=', y.item(), 'dy/dx=', x.grad.item())
x2 = torch.tensor(2.0, requires_grad=True)
z = x2**2 + x2**2 / 2
z.backward()
gradient_value = x2.grad.item()

# 实验二：MLP 完整训练循环

**为什么做？** 这个实验把上一节的单个梯度扩展到一个有两层 Linear 的分类网络。每个参数都由 Autograd 获得梯度，再由 Adam 更新。

代码依次完成：分层划分数据；只用训练集统计量标准化；建立 `4 → 16 → 3` 网络；训练 250 轮；在测试集上推理。

训练循环的固定语义是：`zero_grad` 清除上轮梯度，Forward 产生 logits，Loss 衡量预测错误，Backward 求梯度，`step` 更新参数。

**运行后观察：** Loss 应明显降低，Accuracy 应超过验收线。思考为什么评估阶段需要 `eval()` 和 `no_grad()`，以及为什么标准化不能先看完整数据。

In [ ]:
# 1. 加载 4 个特征、3 个类别的 Iris 数据
features, targets = load_iris(return_X_y=True)

# 2. 分层划分保证三个类别在训练集和测试集中的比例接近
X_train, X_test, y_train, y_test = train_test_split(
    features, targets, test_size=0.25, random_state=42, stratify=targets
)

# 3. 只从训练集学习标准化参数，避免测试信息泄漏
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
train_inputs = torch.tensor((X_train - train_mean) / train_std, dtype=torch.float32)
train_targets = torch.tensor(y_train, dtype=torch.long)
test_inputs = torch.tensor((X_test - train_mean) / train_std, dtype=torch.float32)

# 4. 定义从 4 个输入特征到 3 个类别 logits 的 MLP
model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

# 5. 每轮都完成一次完整的参数更新
for epoch in range(250):
    optimizer.zero_grad()
    logits = model(train_inputs)
    loss = loss_fn(logits, train_targets)
    loss.backward()
    optimizer.step()

# 6. 评估阶段不记录梯度，只把最大 logit 对应的位置作为类别
model.eval()
with torch.no_grad():
    test_logits = model(test_inputs)
    mlp_predictions = test_logits.argmax(dim=1).numpy()
mlp_targets = y_test
print('final loss =', loss.item())
print('test accuracy =', accuracy_score(mlp_targets, mlp_predictions))

# 实验三：CNN 中的 Shape

**为什么做？** CNN 中大量错误来自对 Tensor 维度缺少直觉。这里不训练模型，只执行一次 Forward，把每一层后的 Shape 打印出来。

输入 `[8,1,28,28]` 表示 8 张单通道 28×28 图像。第一层卷积把 Channel 从 1 变成 4；池化把空间尺寸减半；第二层卷积把 Channel 变成 8；全局池化和 Flatten 把特征整理给 Linear 分类头。

**运行后观察：** ReLU 不改变 Shape，卷积和池化各自改变哪些维度？最终 `[8,10]` 中的 10 为什么是类别 logits 而不是图像宽度？

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(output_size=1),
    nn.Flatten(),
    nn.Linear(in_features=8, out_features=10),
)

# 8 张单通道 28×28 模拟图像
value = torch.randn(8, 1, 28, 28)
print('Input:', tuple(value.shape))

# 逐层执行，而不是直接 cnn(value)，以便观察数据流
for layer in cnn:
    value = layer(value)
    print(f'{type(layer).__name__:20s}', tuple(value.shape))

cnn_output = value.detach().numpy()

# 统一验收

最后一段代码把实验结果交给独立验收器：它检查复合函数梯度、MLP 测试准确率和 CNN 输出 Shape。这样“实验成功”不依赖肉眼判断或 Notebook 是否无报错，而是留下结构化证据。

In [ ]:
from validate import validate_exercise, print_report, save_report
report=validate_exercise(globals()); print_report(report)
if report['passed']: save_report(report,Path('artifacts/reference_result.json'))